<a href="https://colab.research.google.com/github/Atharv-1905/Deep-Learning/blob/main/VGG19(Transfer_Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the Kaggle library
!pip install -q kaggle

# 2. Upload the kaggle.json file you just downloaded
from google.colab import files
files.upload()

# 3. Create a kaggle directory and move the file there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# 4. Set permissions so the API can read the file
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
# Example: Replace this with your copied API command
!kaggle datasets download -d akash2sharma/tiny-imagenet

# Unzip the downloaded file
import zipfile
import os

# Find the name of the zip file (usually the last part of the dataset name)
zip_file = "tiny-imagenet.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('./data') # Extracts into a 'data' folder

print("Dataset unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/akash2sharma/tiny-imagenet
License(s): unknown
100% 474M/474M [00:04<00:00, 108MB/s]

Dataset unzipped successfully!


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 64

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Using 20% for validation
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 80000 images belonging to 200 classes.


In [ ]:
val_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 20000 images belonging to 200 classes.


In [ ]:
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
base_model.trainable = False

In [ ]:
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(200, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 2, 2, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 200)            │       102,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,176,072 (80.78 MB)

 Trainable params: 1,151,688 (4.39 MB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [ ]:
history = model.fit(
    train_generator,       # Your training data
    epochs=15,             # Number of passes through data
    validation_data=val_generator, # Data the model hasn't 'seen'
    verbose=1
)

Epoch 1/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 199s 154ms/step - accuracy: 0.0359 - loss: 4.9858 - val_accuracy: 0.1081 - val_loss: 4.4725
Epoch 2/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 182s 145ms/step - accuracy: 0.0835 - loss: 4.4658 - val_accuracy: 0.1432 - val_loss: 4.1373
Epoch 3/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 185s 148ms/step - accuracy: 0.1117 - loss: 4.2283 - val_accuracy: 0.1613 - val_loss: 3.9686
Epoch 4/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 180s 144ms/step - accuracy: 0.1290 - loss: 4.0913 - val_accuracy: 0.1740 - val_loss: 3.8710
Epoch 5/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 180s 144ms/step - accuracy: 0.1422 - loss: 3.9938 - val_accuracy: 0.1807 - val_loss: 3.7990
Epoch 6/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 180s 144ms/step - accuracy: 0.1523 - loss: 3.9180 - val_accuracy: 0.1969 - val_loss: 3.7224
Epoch 7/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 182s 146ms/step - accuracy: 0.1633 - loss: 3.8611 - val_accuracy: 0.2009 - val_loss: 3.6804
Epoch 8/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 183s 147ms/step - ac

**The model has not converged here because VGG models need more epochs to reach convergence, And thats why the accuracy is low**

In [ ]:
test_loss, test_acc = model.evaluate(val_generator)
print(f"Final Accuracy: {test_acc * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 121ms/step - accuracy: 0.2311 - loss: 3.4919
Final Accuracy: 23.11%


#Code to save training progress on drive

In [ ]:
# =================================================================
# 1. MOUNT GOOGLE DRIVE & AUTHENTICATE KAGGLE
# =================================================================
from google.colab import drive, files
import os
import zipfile

# Mount Drive to store checkpoints permanently
drive.mount('/content/drive')

# Create a folder in your Drive for this project if it doesn't exist
checkpoint_dir = "/content/drive/MyDrive/VGG19_TinyImageNet_Project"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# Upload kaggle.json
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json:")
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download and Unzip Tiny ImageNet
!kaggle datasets download -d akash2sharma/tiny-imagenet
with zipfile.ZipFile("tiny-imagenet.zip", 'r') as zip_ref:
    zip_ref.extractall("./data")

# =================================================================
# 2. DATA PREPROCESSING (Generators)
# =================================================================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (64, 64)
BATCH_SIZE = 32

# Data Augmentation to prevent overfitting
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2 # 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# =================================================================
# 3. BUILD VGG19 TRANSFER LEARNING MODEL
# =================================================================
from tensorflow.keras.applications import VGG19
from tensorflow.keras import models, layers, optimizers

# Load pre-trained base
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(64, 64, 3))
base_model.trainable = False # Freeze weights

# Add custom classification head
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(200, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =================================================================
# 4. TRAINING WITH DRIVE CHECKPOINTS
# =================================================================
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Path to save the best model in your Google Drive
# '{epoch:02d}' allows you to see progress epoch by epoch in your Drive
checkpoint_path = os.path.join(checkpoint_dir, "vgg19_best_model.keras")

callbacks = [
    # Stops if validation loss doesn't improve for 5 epochs
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),

    # Saves only the best performing version to your Drive
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print(f"Training started. Checkpoints will be saved to: {checkpoint_path}")

history = model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator,
    callbacks=callbacks
)

# =================================================================
# 5. EVALUATION
# =================================================================
test_loss, test_acc = model.evaluate(val_generator)
print(f"\nFinal Validation Accuracy: {test_acc*100:.2f}%")

#Retrain the model from where it stops


In [ ]:
from tensorflow.keras.models import load_model
import os

# 1. Mount Drive (if not already mounted in this session)
from google.colab import drive
drive.mount('/content/drive')

# 2. Path to your saved checkpoint
# Change this to the exact name of the file in your Drive
checkpoint_path = "/content/drive/MyDrive/VGG19_TinyImageNet_Project/vgg19_best_model.keras"

if os.path.exists(checkpoint_path):
    print("Loading existing model...")
    model = load_model(checkpoint_path)

    # 3. Re-run your Data Generators
    # (The model needs data to continue training!)
    # [Insert your train_generator and val_generator code here]

    # 4. Resume Training
    # initial_epoch=8 means the next epoch will be labelled 'Epoch 9'
    history = model.fit(
        train_generator,
        initial_epoch=8,
        epochs=30,      # The total number of epochs you want to reach
        validation_data=val_generator,
        callbacks=callbacks # Re-use your EarlyStopping/Checkpoint callbacks
    )
else:
    print("Checkpoint file not found. Check your file path!")